# Qwen3-1.7B Unified Sinhala Correction

This is the single training workflow for the project. It combines:

- 300 spoken/ASR-style rows derived from 25 seed phrases.
- 200 unique general-Sinhala rows.

It trains one LoRA adapter for Sinhala transcript and text correction. The
current rows are synthetic and marked `pending`, so this run is a pilot until
a fluent Sinhala reviewer approves or revises the data.

Before running, select **Runtime > Change runtime type > T4 GPU**.


## 1. Verify the GPU


In [ ]:
!nvidia-smi


## 2. Install training dependencies


In [ ]:
!pip -q install -U "transformers>=4.51.0" "datasets>=3.0.0"   "accelerate>=1.0.0" "peft>=0.14.0" "trl>=0.18.0"   "bitsandbytes>=0.49.2" sentencepiece huggingface_hub


## 3. Upload the unified dataset and trainer


In [ ]:
from google.colab import files

uploaded = files.upload()
required = {
    "sinhala_unified_correction_500.jsonl",
    "train_qwen3_unified_sinhala.py",
}
missing = required - set(uploaded)
if missing:
    raise ValueError(f"Upload both required files. Missing: {sorted(missing)}")
print("Ready:", sorted(required))


## 4. Train one unified adapter


In [ ]:
!python /content/train_qwen3_unified_sinhala.py   --dataset /content/sinhala_unified_correction_500.jsonl   --output-dir /content/qwen3-1.7b-unified-sinhala   --epochs 3   --max-length 256   --allow-pending


## 5. Inspect the evaluation metrics


In [ ]:
import json

with open(
    "/content/qwen3-1.7b-unified-sinhala/test_metrics.json",
    encoding="utf-8",
) as file:
    metrics = json.load(file)

print(json.dumps(metrics, ensure_ascii=False, indent=2))


## 6. Download the adapter and evaluation files


In [ ]:
from google.colab import files

for path in [
    "/content/qwen3-1.7b-unified-sinhala/final-adapter.zip",
    "/content/qwen3-1.7b-unified-sinhala/test_metrics.json",
    "/content/qwen3-1.7b-unified-sinhala/test_predictions.csv",
]:
    files.download(path)
